In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('CASQL.csv')

# 1. Print the column names so we can map our logic perfectly
print("--- COLUMN NAMES ---")
print(df.columns.tolist())
print("\n")

# 2. Check for missing values and data types
print("--- DATA INFO ---")
print(df.info())
print("\n")

# 3. Look at the first 3 rows to understand the formatting
print("--- DATA SAMPLE ---")
print(df.head(3))

--- COLUMN NAMES ---
['Customer ID', 'Age', 'Gender', 'Item Purchased', 'Category', 'Purchase Amount (USD)', 'Location', 'Size', 'Color', 'Season', 'Review Rating', 'Subscription Status', 'Shipping Type', 'Discount Applied', 'Promo Code Used', 'Previous Purchases', 'Payment Method', 'Frequency of Purchases']


--- DATA INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900

In [7]:
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,...,promo_dependency_score,satisfaction_flag,ltv_percentile,value_tier,purchase_vol_pct,loyalty_score_A,annual_freq_estimate,freq_pct,sub_flag,loyalty_score_B
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,1.0,0,0.296923,Bronze,0.271923,0.331872,26.0,0.722308,1,0.833385
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,1.0,0,0.051154,Bronze,0.030641,0.238692,26.0,0.722308,1,0.833385
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,1.0,0,0.637308,Silver,0.451026,0.535026,52.0,0.931026,1,0.958615
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,1.0,0,0.986410,Platinum,0.972949,0.929872,52.0,0.931026,1,0.958615
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,1.0,0,0.578846,Silver,0.615385,0.517846,1.0,0.148333,1,0.489000


In [3]:
df.describe()


,Customer ID,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.000000,3900.000000,3900.000000,3863.000000,3900.000000
mean,1950.500000,44.068462,59.764359,3.750065,25.351538
std,1125.977353,15.207589,23.685392,0.716983,14.447125
min,1.000000,18.000000,20.000000,2.500000,1.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000


In [4]:
import pandas as pd
import numpy as np

# 1. LOAD THE DATA
df = pd.read_csv('CASQL.csv')

# 2. CLEANING & IMPUTATION
# Fill the 37 missing Review Ratings with the median rating
median_rating = df['Review Rating'].median()
df['Review Rating'] = df['Review Rating'].fillna(median_rating)

# 3. BASE METRICS & PROXIES
# Create an Estimated Lifetime Value (LTV) proxy
df['Estimated LTV'] = (df['Previous Purchases'] + 1) * df['Purchase Amount (USD)']

# 4. PROMO DEPENDENCY SCORE
# Convert Yes/No to 1/0 for math
df['discount_flag'] = np.where(df['Discount Applied'] == 'Yes', 1, 0)
df['promo_flag'] = np.where(df['Promo Code Used'] == 'Yes', 1, 0)
# Score from 0.0 to 1.0
df['promo_dependency_score'] = (df['discount_flag'] + df['promo_flag']) / 2.0

# 5. SATISFACTION FLAG
# 1 = Satisfied (Rating >= 4.0), 0 = Unsatisfied
df['satisfaction_flag'] = np.where(df['Review Rating'] >= 4.0, 1, 0)

# 6. CUSTOMER VALUE TIERS
# Segmenting based on Estimated LTV for the Power BI Dashboard
df['ltv_percentile'] = df['Estimated LTV'].rank(pct=True)

conditions = [
    (df['ltv_percentile'] >= 0.90),
    (df['ltv_percentile'] >= 0.70) & (df['ltv_percentile'] < 0.90),
    (df['ltv_percentile'] >= 0.40) & (df['ltv_percentile'] < 0.70),
    (df['ltv_percentile'] < 0.40)
]
df['value_tier'] = np.select(conditions, ['Platinum', 'Gold', 'Silver', 'Bronze'], default='Bronze')

# 7. LOYALTY HYPOTHESIS A: TRANSACTIONAL (Volume & Spend)
df['purchase_vol_pct'] = df['Previous Purchases'].rank(pct=True)
# Blending Volume (60%) and Current Spend (40%)
df['loyalty_score_A'] = (df['purchase_vol_pct'] * 0.6) + (df['Purchase Amount (USD)'].rank(pct=True) * 0.4)

# 8. LOYALTY HYPOTHESIS B: BEHAVIORAL (Engagement & Subscription)
# Map text frequency to an estimated numerical annual frequency
freq_mapping = {
    'Weekly': 52, 
    'Fortnightly': 26, 
    'Monthly': 12, 
    'Quarterly': 4, 
    'Annually': 1,
    'Bi-Weekly': 26
}
# Apply mapping, default to 1 if unknown string appears
df['annual_freq_estimate'] = df['Frequency of Purchases'].map(freq_mapping).fillna(1)
df['freq_pct'] = df['annual_freq_estimate'].rank(pct=True)

df['sub_flag'] = np.where(df['Subscription Status'] == 'Yes', 1, 0)
# Blending Frequency (60%) and Subscription Status (40%)
df['loyalty_score_B'] = (df['freq_pct'] * 0.6) + (df['sub_flag'] * 0.4)

# 9. HYPOTHESIS TESTING
# Which loyalty definition better predicts our proxy for Lifetime Value?
print("--- LOYALTY HYPOTHESIS TESTING ---")
corr_matrix = df[['loyalty_score_A', 'loyalty_score_B', 'Estimated LTV']].corr()
print(corr_matrix['Estimated LTV'])

# 10. CLEAN UP AND EXPORT FOR SQL
# Drop intermediate calculation columns to keep the dataset clean
columns_to_drop = ['discount_flag', 'promo_flag', 'ltv_percentile', 'purchase_vol_pct', 'annual_freq_estimate', 'freq_pct', 'sub_flag']
df_final = df.drop(columns=columns_to_drop)

# Save the final engineered dataset
df_final.to_csv('engineered_CASQL.csv', index=False)
print("\nSuccess! Saved as 'engineered_CASQL.csv'")

--- LOYALTY HYPOTHESIS TESTING ---
loyalty_score_A    0.951945
loyalty_score_B    0.013122
Estimated LTV      1.000000
Name: Estimated LTV, dtype: float64

Success! Saved as 'engineered_CASQL.csv'
